# Configuration

In [3]:
import os 
import pickle 

if True ^ os.getcwd().endswith("winners-curse"):
    os.chdir('..')

from typing import Iterable, Union
from joblib import Parallel, delayed

In [4]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [5]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# visualization params
# label size
tick_label_size = 12
legend_label_size = 12
axis_label_size = 14
title_size = 18
# font
plt.rcParams['font.family'] = 'serif'

# model label map 
model_label_map = {
    'plugin': 'Plugin', 
    'standard_bootstrap': 'Standard Bootstrap', 
    'mn_bootstrap': 'm-out-of-n Bootstrap', 
    'num_bootstrap': 'Numerical Bootstrap'
}

In [6]:
from core.dgp import DataGenerationProcess
from core.variables import *

# DGP

In [7]:
sample_size = 1000

customer_arr = np.random.uniform(-1, 1, size=sample_size)

In [ ]:
class Targeting(DataGenerationProcess):
    valid_response_types = ['continuous', 'bernoulli', 'logit']
    def __init__(
        self, 
        base_effects: list[np.ndarray], 
        n_segments: int, 
        segment_func: callable, 
        noise_vars: list[ContinuousRandomVariable] = None, 
        response_type: str = 'continuous', 
        dgp_seed: int = 0
    ):
        """
        Data generation process for multiple segments with multiple treatments.

        Params:
        -------
        base_effects: list[np.ndarray]
            List of base treatment effects for each segment.
        n_segments: int
            Number of segments.
        segment_func: callable
            Function to generate segment values.
        noise_vars: list[ContinuousRandomVariable]
            List of noise variables for each segment.
        noise_std: float
            Standard deviation of the noise.
        response_type: str
            Type of response variable. Can be 'continuous', 'bernoulli', or 'logit'.
        """
        if dgp_seed is not None:
            np.random.seed(dgp_seed)

        assert response_type in self.valid_response_types, "Response type should be continuous, bernoulli, or logit."
        if response_type == 'continuous':
            assert noise_vars is not None, "Noise standard deviation should be provided for continuous response type."
        
        self.base_effects = base_effects  # shape = (n_treatments, )
        self.segment_func = segment_func 
        self.n_segments = n_segments 
        self.noise_vars = noise_vars
        self.response_type = response_type

        # extract attributes
        self.n_treatments = len(base_effects)
        self.segment_idx_arr = np.arange(self.n_segments)
        self.treatment_idx_arr = np.arange(self.n_treatments)

        # draw true effects
        self.treatment_effects = self.draw_true_effects(seed=dgp_seed)

    def draw_true_effects(self, seed: int = None):
        if seed is not None:
            np.random.seed(seed)

        return np.concatenate([
            prior_dstn.sample(self.n_segments)[np.newaxis, :]  # shape = (1, n_segments)
            for prior_dstn in self.base_effects
        ], axis=0)  # shape = (n_arms, n_segments)

    def sample_individuals(self, sample_size: int, seed: int = None) -> np.ndarray:
        """ 
        Sample individuals from the DGP. 
        """
        if seed is not None:
            np.random.seed(seed)

        return np.random.uniform(0, 1, size=(sample_size, ))

    def sample(self, sample_size, seed: int = None) -> tuple:
        """ 
        Sample data from the DGP. 

        Params:
        -------
        sample_size: int
            Number of individuals to sample.
        seed: int
            Random seed for reproducibility.

        Returns:
        --------
        tuple: 
            - segments: np.ndarray, shape = (sample_size, )
                Array of sampled individuals' segments.
            - treatments: np.ndarray, shape = (sample_size, )
                Array of sampled individuals' treatments.
            - outcomes: np.ndarray, shape = (sample_size, )
                Array of sampled individuals' outcomes.
        """
        if seed is not None:
            np.random.seed(seed)

        # sample customer segments
        segment_arr = self.sample_individuals(sample_size)

        # sample treatments
        treatment_arr = np.random.choice(self.treatment_idx_arr, size=sample_size)

        # compute outcomes
        outcome_arr = self.outcome_generation(segment_arr, treatment_arr, seed)

        return segment_arr, treatment_arr, outcome_arr
    
    def outcome_generation(self, segments: np.ndarray, treatments: np.ndarray, seed: int = None) -> np.ndarray:
        """ 
        Generate outcomes for the given sample size. 

        Params:
        -------
        segments: np.ndarray, shape = (sample_size, )
            Array of segment values.
        treatments: np.ndarray, shape = (sample_size, )
            Array of treatment values.
        seed: int
            Random seed for reproducibility.

        Returns:
        --------
        np.ndarray, shape = (sample_size, )
            Array of outcomes.
        """
        if seed is not None:
            np.random.seed(seed)

        treatment_effect_arr = self.te_arr[segments, treatments]  # shape = (sample_size, )

        if self.response_type == 'continuous':
            noises = np.random.normal(loc=0, scale=self.noise_std, size=(segments.shape[0], ))
            return treatment_effect_arr + noises
        elif self.response_type == 'bernoulli':  # bernoulli response
            return np.random.binomial(n=1, p=treatment_effect_arr)
        elif self.response_type == 'logit':
            proba_arr = 1 - expit(-treatment_effect_arr)
            return np.random.binomial(n=1, p=proba_arr)
        else: # invalid response types
            raise ValueError("Invalid response type. Choose from {}.".format(self.valid_response_types))

In [25]:
def equal_bin_segment(n_segments: int, customers: np.ndarray) -> np.ndarray:
    assert len(customers.shape) == 1, "Customer array should be 1D."

    return pd.cut(customers, bins=n_segments, labels=np.arange(n_segments)).to_numpy()